# 02. Conventional PBMC Cohort Walkthrough (Multi-Batch, Controls & DA/DS)

In this tutorial, you will analyze a **12-sample longitudinal human PBMC cohort** (`quintelier_2023_cytonorm_meso`: 4 patients across 3 vaccination timepoints and 2 clinical acquisition batches), then inspect two targeted complements that exercise capabilities absent from pre-compensated PBMC files:
1. **`arguello_2020_scenith_metabolism` (PBMC Single-Stain Control Complement):** 24-color human PBMC panel calculating spillover directly from `raw/controls/` single-stained tubes.
2. **`li_2025_caerin_macrophage_ly6c` (Non-PBMC Full-Cascade DA/DS Complement):** 12-sample cohort ($n=4$/group) exercising simultaneous single-stain controls + viability (`BV510-A`) + dump (`APC-Cy7-A`) + statistically significant DA & DS hits.

> **Prerequisites (Download Raw `.fcs` Data):** Curated `metadata/` CSVs are included in the repository. Before running this notebook, download the raw `.fcs` files once from the terminal:
> ```bash
> flowx download quintelier_2023_cytonorm_meso arguello_2020_scenith_metabolism li_2025_caerin_macrophage
> ```

In [1]:
import os
from pathlib import Path

# Ensure working directory is the repository root whether launched from root or docs/tutorials/
if not Path("configs/config_run").exists() and Path("../../configs/config_run").exists():
    os.chdir("../..")
print("Working directory ready:", Path("configs/config_run").exists())

Working directory ready: True


## 1. Inspect the Longitudinal PBMC Cohort Design (`quintelier_2023_cytonorm_meso`)

Twelve human PBMC samples span 3 conditions (`Baseline_Pre_Vaccination`, `Post_Vaccine_Dose_1`, `Post_Vaccine_Dose_3`) across 2 acquisition dates (`2020-08-10_LSRFortessa` and `2022-06-16_LSRFortessa`).

In [2]:
import pandas as pd

samples_df = pd.read_csv("data/conventional/quintelier_2023_cytonorm_meso/metadata/samples.csv")
print(samples_df[["sample_id", "condition", "batch_id", "is_control"]].to_string(index=False))

     sample_id                condition               batch_id  is_control
ID1_Panel1_TP1 Baseline_Pre_Vaccination 2020-08-10_LSRFortessa       False
ID1_Panel1_TP2      Post_Vaccine_Dose_1 2020-08-10_LSRFortessa       False
ID1_Panel1_TP3      Post_Vaccine_Dose_3 2020-08-10_LSRFortessa       False
ID2_Panel1_TP1 Baseline_Pre_Vaccination 2020-08-10_LSRFortessa       False
ID2_Panel1_TP2      Post_Vaccine_Dose_1 2020-08-10_LSRFortessa       False
ID2_Panel1_TP3      Post_Vaccine_Dose_3 2020-08-10_LSRFortessa       False
ID3_Panel1_TP1 Baseline_Pre_Vaccination 2020-08-10_LSRFortessa       False
ID3_Panel1_TP2      Post_Vaccine_Dose_1 2020-08-10_LSRFortessa       False
ID3_Panel1_TP3      Post_Vaccine_Dose_3 2020-08-10_LSRFortessa       False
ID5_Panel1_TP1 Baseline_Pre_Vaccination 2022-06-16_LSRFortessa       False
ID5_Panel1_TP2      Post_Vaccine_Dose_1 2022-06-16_LSRFortessa       False
ID5_Panel1_TP3      Post_Vaccine_Dose_3 2022-06-16_LSRFortessa       False


## 2. Run the Unified Cohort Pipeline

```bash
flowx pipeline -c configs/config_run/conventional/quintelier_2023_cytonorm_meso.yaml --overwrite
```

Because `quintelier_2023_cytonorm_meso` contains pre-gated, pre-transformed PBMC FCS files, `01_time_qc`–`04_transformation` are disabled in the YAML while `04_batch_preparation`, `05_population_discovery` (`marker_roles: [lineage, functional]`), and `06_statistical_analysis` (`Baseline_Pre_Vaccination` vs `Post_Vaccine_Dose_3`) run end-to-end.

In [3]:
import json
from pathlib import Path

out_dir = Path("outputs/config_run/conventional/quintelier_2023_cytonorm_meso")
ba = json.loads((out_dir / "04_batch_preparation/tables/batch_necessity_assessment.json").read_text())
print("Recommendation:", ba["recommendation"])
print("Drift Score:", ba["drift_score"], "/ 100.0")
print("Batches:", ba["unique_batches"])

b_inv = pd.read_csv(out_dir / "04_batch_preparation/tables/batch_invariance_metrics.csv")
print(f"Mean EMD reduction across {len(b_inv)} markers: {b_inv['emd_reduction_pct'].mean():+.2f}%")

Recommendation: CAUTION_ANCHOR_FREE
Drift Score: 100.0 / 100.0
Batches: ['2020-08-10_LSRFortessa', '2022-06-16_LSRFortessa']
Mean EMD reduction across 14 markers: +99.55%


![PBMC Batch Density Alignment (Quintelier 2023)](../images/tutorials/batch_density_pbmc_quintelier.png)

## 3. Step 06 Differential State (DS) Testing (`Baseline` vs `Post_Vaccine_Dose_3`)

With $n=4$ biological replicates per timepoint, non-parametric Mann-Whitney testing detects functional marker shifts (`p = 0.0286`) across PBMC subsets:

In [4]:
ds_df = pd.read_csv(out_dir / "06_statistical_analysis/tables/differential_state_summary.csv")
print(ds_df.head(6)[["population", "marker", "mean_ref", "mean_target", "delta_stat", "cohens_d", "p_value"]].to_string(index=False))

                                population     marker  mean_ref  mean_target  delta_stat  cohens_d  p_value
            Unassigned (IgM++ IgD++ CD5++)       PE-A    1.5513       1.6220      0.0707    1.9055 0.057143
           Unassigned (IgD++ PDL1++ IgM++)    BV605-A    3.3850       3.4501      0.0651    1.5537 0.114286
Switched Memory B Cells (CD19+ IgD- CD27+)    BV605-A    1.4414       1.5083      0.0670    1.4824 0.114286
          Unassigned (CD27++ IgM++ CD24++)    BB515-A    0.9474       1.0526      0.1051    1.3633 0.200000
           Unassigned (IgM++ CD86++ CD5++) PE-CF594-A    1.1640       1.4502      0.2862    1.1074 0.114286
          Unassigned (CD27++ IgM++ CD24++)    BV605-A    3.4093       3.5575      0.1482    1.0884 0.342857


![PBMC Differential State Heatmap (Quintelier 2023)](../images/tutorials/stats_ds_heatmap_pbmc_quintelier.png)

## 4. PBMC Complement: Single-Stain Control Spillover (`arguello_2020_scenith_metabolism`)

To see FlowX calculate a $24 \times 24$ spillover matrix directly from single-stained PBMC controls (`raw/controls/`) and resolve 12 recognizable PBMC populations (`CD14+ Classical Monocytes`, `NK Cells (CD56+)`, `Switched/Unswitched/Transitional B Cells`), inspect `arguello_2020_scenith_metabolism`:

```bash
flowx pipeline -c configs/config_run/conventional/arguello_2020_scenith_metabolism.yaml --overwrite
```

In [5]:
arg_dir = Path("outputs/config_run/conventional/arguello_2020_scenith_metabolism")
arg_audit = pd.read_csv(arg_dir / "02_compensation/tables/control_gating_and_qc_audit.csv")
print("24-Color PBMC Controls Audited:", len(arg_audit))
print(arg_audit.head(5).to_string(index=False))

arg_pops = pd.read_csv(arg_dir / "05_population_discovery/tables/cluster_to_population_map.csv")
print("\nDiscovered PBMC Populations:", sorted(arg_pops["Annotated_Population"].unique()))

24-Color PBMC Controls Audited: 23
channel carrier_type method  n_initial  debris_removed_events  debris_removed_pct  doublets_removed_events  doublets_removed_pct  n_events  n_clean_events  scatter_singlet_kept_pct  n_pos  n_neg   pos_mfi  neg_mfi    neg_rsd  delta_mfi    snr  stain_index  pos_fsc_median  neg_fsc_median                                                                        control_file              marker  raw_events  time_qc_passed_events  time_qc_passed_pct time_qc_status         time_qc_action       estimator_used  max_residual_abs_z max_residual_channel  max_residual_abs_pct      residual_status
 B525-A        beads    gmm        868                    160               18.43                      139                 16.01       569             569                     65.55    462     94 4495249.5 -14158.0 12714.7776  4509407.5 354.66       177.33       1124835.5       1114625.0       Scenith_05062024_Compensation Specimen_A1_B525 AF488 - Puromycin_20240521.fcs AF4

![24-Color PBMC Spillover Matrix (Arguello 2020)](../images/tutorials/conv_spillover_matrix_arguello.png)

![24-Color PBMC Control Gating Audit (Arguello 2020)](../images/tutorials/conv_control_gating_arguello.png)

## 5. Non-PBMC Complement: Full-Cascade Controls + Dump + DA/DS (`li_2025_caerin_macrophage_ly6c`)

When you want a single cohort that executes **all 6 steps simultaneously**—single-stain controls, `BV510-A` viability, `APC-Cy7-A` dump exclusion, and $n=4$/group Differential Abundance (`p = 0.0286`) & Differential State (`p = 0.0211`)—run `li_2025_caerin_macrophage_ly6c`:

In [6]:
li_dir = Path("outputs/config_run/conventional/li_2025_caerin_macrophage_ly6c")
li_da = pd.read_csv(li_dir / "06_statistical_analysis/tables/differential_abundance_summary.csv")
print(li_da.head(4)[["population", "mean_ref_pct", "mean_target_pct", "log2fc", "p_value", "q_value"]].to_string(index=False))

                                  population  mean_ref_pct  mean_target_pct  log2fc  p_value  q_value
Murine Inflammatory Monocytes (Ly6C+ CD11b+)        2.1919           0.5972 -1.8586 0.028571 0.100000
                                    CD45.2++        1.0620           0.2307 -2.1549 0.028571 0.100000
                      Myeloid Cells (CD11b+)        1.8798           1.0416 -0.8455 0.057143 0.133333
                         Unassigned (CD11c+)       56.1923          64.5731  0.2005 0.114286 0.200000


## Next Steps

- Continue to **[03. Spectral 32-Color PBMC Cohort Walkthrough](03_spectral_pbmc_cohort_walkthrough.ipynb)** for high-dimensional full-spectrum PBMC phenotyping.

## References & Scientific Standards

* **MIFlowCyt Standard:** [](doi:10.1002/cyto.a.20623)
* **FCS 3.1 Data File Standard:** [](doi:10.1002/cyto.a.20825)
* **Gating-ML 2.0 Standard:** [](doi:10.1002/cyto.a.22690)
* **FlowSOM Clustering:** [](doi:10.1002/cyto.a.22625)
* **CytoNorm Batch Normalization:** [](doi:10.1002/cyto.a.23904)
* **UMAP Dimensionality Reduction:** [](doi:10.48550/arXiv.1802.03426)
* **FlowKit Computational Toolkit:** [](doi:10.3389/fimmu.2021.768541)